<a href="https://colab.research.google.com/github/delsucflorian/Oncolake_TorchProtein/blob/main/notebooks/03_prostt5_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:

!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [2]:
import os
from google.colab import drive
drive.mount('/content/drive')
!pip install transformers sentencepiece -q
!pip install biopython -q

if not os.path.exists("/content/data/alphafold"):

  !mkdir -p /content/data
  !cp -r "/content/drive/MyDrive/oncolake_torchprotein/data/alphafold" /content/data/
else :
  print("Rien a installer coté drive ")
if not os.path.exists("/content/foldseek/bin/foldseek"):
  !wget https://mmseqs.com/foldseek/foldseek-linux-avx2.tar.gz -q
  !tar xvfz foldseek-linux-avx2.tar.gz > /dev/null
  !chmod +x foldseek/bin/foldseek
  !foldseek/bin/foldseek version
else:
  print("Rien a installer coté foldseek")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 4.6 MB/s eta 0:00:00
463739e0014a1549a527de589102cde98f802f37


In [3]:
import transformers
print(f"Transformers : {transformers.__version__}")

Transformers : 5.16.1


In [4]:
!mkdir -p /content/foldseek_output
!foldseek/bin/foldseek createdb /content/data/alphafold /content/foldseek_output/db


createdb /content/data/alphafold /content/foldseek_output/db 

MMseqs Version:         	463739e0014a1549a527de589102cde98f802f37
Use GPU                 	0
Path to ProstT5         	
Chain name mode         	0
Model name mode         	0
Write mapping file      	0
Write Foldcomp          	0
Mask b-factor threshold 	0
Coord store mode        	2
Save residue indices    	false
Write lookup file       	1
Input format            	0
Input compression format	0
File Inclusion Regex    	.*
File Exclusion Regex    	^$
Threads                 	2
Verbosity               	3

Output file: /content/foldseek_output/db
[=================================================================] 100.00% 410 19s 724ms
Time for merging to db_ss: 0h 0m 0s 0ms
Time for merging to db_h: 0h 0m 0s 0ms
Time for merging to db_ca: 0h 0m 0s 10ms
Time for merging to db: 0h 0m 0s 0ms
Ignore 0 out of 410.
Too short: 0, incorrect: 0, not proteins: 0.
Time for processing: 0h 0m 19s 823ms


In [5]:
!ls -la /content/foldseek_output/

total 2152
drwxr-xr-x 2 root root    4096 Sep 12 12:09 .
drwxr-xr-x 1 root root    4096 Sep 12 12:09 ..
-rw-r--r-- 1 root root  262983 Sep 12 12:09 db
-rw-r--r-- 1 root root 1576258 Sep 12 12:09 db_ca
-rw-r--r-- 1 root root       4 Sep 12 12:09 db_ca.dbtype
-rw-r--r-- 1 root root    6541 Sep 12 12:09 db_ca.index
-rw-r--r-- 1 root root       4 Sep 12 12:09 db.dbtype
-rw-r--r-- 1 root root   16705 Sep 12 12:09 db_h
-rw-r--r-- 1 root root       4 Sep 12 12:09 db_h.dbtype
-rw-r--r-- 1 root root    4938 Sep 12 12:09 db_h.index
-rw-r--r-- 1 root root    5932 Sep 12 12:09 db.index
-rw-r--r-- 1 root root    5934 Sep 12 12:09 db.lookup
-rw-r--r-- 1 root root    4393 Sep 12 12:09 db.source
-rw-r--r-- 1 root root  262983 Sep 12 12:09 db_ss
-rw-r--r-- 1 root root       4 Sep 12 12:09 db_ss.dbtype
-rw-r--r-- 1 root root    5932 Sep 12 12:09 db_ss.index


In [6]:
!foldseek/bin/foldseek lndb /content/foldseek_output/db_h /content/foldseek_output/db_ss_h
!foldseek/bin/foldseek convert2fasta /content/foldseek_output/db_ss /content/foldseek_output/3di_sequences.fasta

lndb /content/foldseek_output/db_h /content/foldseek_output/db_ss_h 

MMseqs Version:	463739e0014a1549a527de589102cde98f802f37
Verbosity	3

Time for processing: 0h 0m 0s 0ms
convert2fasta /content/foldseek_output/db_ss /content/foldseek_output/3di_sequences.fasta 

MMseqs Version:	463739e0014a1549a527de589102cde98f802f37
Use header DB	false
Verbosity    	3

Start writing file to /content/foldseek_output/3di_sequences.fasta
Time for processing: 0h 0m 0s 2ms


In [7]:
!head -6 /content/foldseek_output/3di_sequences.fasta
!grep -c "^>" /content/foldseek_output/3di_sequences.fasta

>A0A2R8Y7D0 Ubiquitin domain-containing protein TINCR
DDDDDDDDDPDDWAFEWEQELVVSDTDTDIHGQPDFLLVVCVVCVVVVHPQPLWFKDFLLDTDDRRDGCNNVVNHHHGYIYTDNDSVSVVVCSVVVVVVVVVVVVVVVVVVPPPDPDPDD
>A1YPR0 Zinc finger and BTB domain-containing protein 7C
DPPPPPDPDDDDDPCPVQVVLQVQVVCQVVLHPFQEWEDEPHDIGGHDLVLLLVQFPQSVVVVVPDPDDDHRDYFYDDQADPVLVVQSVCCSRNVKGWDDPVCLVRVLSVCVVRVRVVSVVVSVVVVDPDDDDDDDDDDDPPPPPPCDDPPDPPDDDDDDDDDDDDDDDDDDDDDDDDDDPPPPLPDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDIDTDIDHVVVVVVNVPDDDPDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDPPDPGPPDPDDDDDDDDDDDDDDDDDPPCPVVVVDPDDDDDDDDDDDPPDDPLVVVQVVCPPPVVPPDDNDPPPPDPPVPQQDWDAAPPPRDTDGHPVVNVLVVCSVVVDFPDADPPPRDTHSDVVVVVLVCCVVVVDQPDADPVNRDGHSDPVVNQLVVCSVVVDFPDADPQAGDGHSDPVVSVVCVVVVVSVRPDPPPPDDDPVVVVVCVVDDPDDDDDDDDDPDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDPVVVVVVVVVVVVVVVVVVVVVVVVVVVPDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDDD
>A2RRD8 Zinc finger protein 320
DPPPPDLDDLVNVQDDDDPVVVVVDDPVRVVVSVVSSVVSVVVSVVVVLVVVVVVVVPPDDDDDDDDDDDPDDPDDDDDDDPPPPPVVVNVVVVVVVVVVVVVVVVPPDDDDDDDP

In [8]:
from transformers import T5Tokenizer, T5EncoderModel
import torch

# Nom du modèle sur HuggingFace Hub
MODEL_NAME = "Rostlab/ProstT5"

# Charger le tokenizer (qui convertit texte → tokens numériques)
print("Chargement du tokenizer...")
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME, do_lower_case=False)

# Charger le modèle (l'encodeur uniquement — on n'a pas besoin du décodeur)
print("Chargement du modèle (peut prendre 1-2 min, ~2 GB à télécharger)...")
model = T5EncoderModel.from_pretrained(MODEL_NAME)

# Passer en mode évaluation (désactive dropout, etc.) et sur GPU
model = model.eval().to("cuda")

# Utiliser half-precision pour économiser la VRAM
model = model.half()

print(f"\nModèle chargé sur : {next(model.parameters()).device}")
print(f"Nombre de paramètres : {sum(p.numel() for p in model.parameters())/1e6:.0f}M")

Chargement du tokenizer...


tokenizer_config.json:   0%|          | 0.00/2.60k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  238kB            

spiece.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/283 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 11.3GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
from Bio import SeqIO

import pandas as pd
features_ref = pd.read_parquet('/content/drive/MyDrive/oncolake_torchprotein/data/features_baseline_ref.parquet')
accessions_404 = features_ref['accession'].tolist()
print(f"Nombre d'accessions dans le parquet: {len(accessions_404)}")
FASTA_3DI = '/content/foldseek_output/3di_sequences.fasta'


sequences_3di = {}
for record in SeqIO.parse(FASTA_3DI, "fasta"):
    accession = record.id
    sequence = str(record.seq)
    if accession in accessions_404:
      sequences_3di[accession] = sequence
print(f"Nombre de séquences chargées : {len(sequences_3di)}")

lengths = [len(s) for s in sequences_3di.values()]
print(f"Longueur min : {min(lengths)}")
print(f"Longueur max : {max(lengths)}")
print(f"Longueur médiane : {sorted(lengths)[len(lengths)//2]}")

for i, (acc, seq) in enumerate(list(sequences_3di.items())[:3]):
    print(f"\n{acc} (longueur {len(seq)})")
    print(f"  {seq[:80]}...")
missing = set(accessions_404) - set(sequences_3di.keys())
print(f"Accessions manquantes ({len(missing)}) : {missing}")

In [ ]:
from tqdm import tqdm
import numpy as np
import torch

def get_embedding(seq_3di, model, tokenizer, device='cuda', max_length=1500):
    """
    Retourne un vecteur numpy (1024,) qui représente la protéine.
    """
    seq = seq_3di
    if len(seq) > max_length:
        seq = seq[:max_length]
    seq = '<fold2AA> ' + ' '.join(seq.lower())
    tokens = tokenizer(seq, return_tensors='pt', padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = model(input_ids=tokens['input_ids'], attention_mask=tokens['attention_mask'])
    outputs = outputs.last_hidden_state
    final_output = torch.mean(outputs, dim=1)
    # 8. Retourner un numpy array
    return final_output.cpu().numpy().squeeze()


In [ ]:
test_acc = list(sequences_3di.keys())[0]
test_seq = sequences_3di[test_acc]
emb = get_embedding(test_seq, model, tokenizer)
print(f"Test sur {test_acc}, shape : {emb.shape}")

In [ ]:
embeddings_dict = {}
for accession, seq in tqdm(sequences_3di.items(), desc="Embeddings"):
  embeddings_dict[accession] = get_embedding(seq, model, tokenizer)